<a href="https://colab.research.google.com/github/mirian2004/AI-AI-/blob/week02/week02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
# 1. 로컬 임베딩 모델 로드

print("모델을 다운로드하고 로드중입니다... (최초 1회만 소요)")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("모델 로드 완료!")

모델을 다운로드하고 로드중입니다... (최초 1회만 소요)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

모델 로드 완료!


In [4]:
# 2. 데이터 준비 (지식 베이스)

sentences = [
    "소화가 잘 안되고 배가 아파요.",
    "오늘 점심 메뉴로 김치찌개 어때?",
    "파이썬 프로그래밍은 정말 재미있어",
    "머리가 깨질 듯이 아프고 열이 나요.",
    "주식 시장이 오늘 폭락했습니다.",
    "강남역 근처 맛집 추천해줘."
]

In [5]:
# 3. 임베딩(Embedding) 생성

print("텍스트를 벡터로 변환 중...")

#임베딩 벡터 공간으로 변환
embeddings = model.encode(sentences)

#FAISS 사용을 위해 floats32 타입으로 변환
embedding_matrix = np.array(embeddings).astype('float32')

#차원 확인
dimension = embedding_matrix.shape[1]
print(f"임베딩 완료! 벡터 차원 수 : {dimension}")

텍스트를 벡터로 변환 중...
임베딩 완료! 벡터 차원 수 : 384


In [6]:
# 4. FAISS 인덱스 구축 (벡터 DB)

# 코사인 유사도 계산을 위해 벡터 정규화(Normalize)
faiss.normalize_L2(embedding_matrix)

#인덱스 생성 및 데이터 추가
index = faiss.IndexFlatIP(dimension)
index.add(embedding_matrix)

In [11]:
# 5. 의미적 검색(Semantic Dearch) 함수

def search(query, k=2):
  print(f"\n검색어: '{query}'")

  # 1. 질문을 벡터로 변환
  query_vec = model.encode([query])
  query_vec = np.array(query_vec).astype('float32')

  # 2. 정규화
  faiss.normalize_L2(query_vec)

  # 3. 검색
  D, I = index.search(query_vec, k)

  # 4. 결과 출력
  for i in range(k):
    idx = I[0][i]
    score = D[0][i]
    print(f"순위 {i+1}: [유사도 {score:.4f}] {sentences[idx]}")


# --- [실습 테스트] ---
# 키워드가 없는 문장으로 테스트해보자

search("AI 공부는 어떻게 하는거지?")
search("맛있는 거 먹고 싶다")


검색어: 'AI 공부는 어떻게 하는거지?'
순위 1: [유사도 0.2358] 파이썬 프로그래밍은 정말 재미있어
순위 2: [유사도 0.0844] 머리가 깨질 듯이 아프고 열이 나요.

검색어: '맛있는 거 먹고 싶다'
순위 1: [유사도 0.8069] 오늘 점심 메뉴로 김치찌개 어때?
순위 2: [유사도 0.5995] 파이썬 프로그래밍은 정말 재미있어


In [12]:
import numpy as np

# 전체 문장 간 코사인 유사도 행렬 확인
# 이미 정규화되어 있으므로 내적 = 코사인 유사도
sim_matrix = embedding_matrix @ embedding_matrix.T

for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if i < j:
            print(f"{sim_matrix[i][j]:.4f} | {s1}  <->  {s2}")

0.2840 | 소화가 잘 안되고 배가 아파요.  <->  오늘 점심 메뉴로 김치찌개 어때?
0.0282 | 소화가 잘 안되고 배가 아파요.  <->  파이썬 프로그래밍은 정말 재미있어
0.4852 | 소화가 잘 안되고 배가 아파요.  <->  머리가 깨질 듯이 아프고 열이 나요.
0.2802 | 소화가 잘 안되고 배가 아파요.  <->  주식 시장이 오늘 폭락했습니다.
0.0998 | 소화가 잘 안되고 배가 아파요.  <->  강남역 근처 맛집 추천해줘.
0.5349 | 오늘 점심 메뉴로 김치찌개 어때?  <->  파이썬 프로그래밍은 정말 재미있어
0.2653 | 오늘 점심 메뉴로 김치찌개 어때?  <->  머리가 깨질 듯이 아프고 열이 나요.
0.1427 | 오늘 점심 메뉴로 김치찌개 어때?  <->  주식 시장이 오늘 폭락했습니다.
0.4726 | 오늘 점심 메뉴로 김치찌개 어때?  <->  강남역 근처 맛집 추천해줘.
0.0663 | 파이썬 프로그래밍은 정말 재미있어  <->  머리가 깨질 듯이 아프고 열이 나요.
0.1219 | 파이썬 프로그래밍은 정말 재미있어  <->  주식 시장이 오늘 폭락했습니다.
0.1854 | 파이썬 프로그래밍은 정말 재미있어  <->  강남역 근처 맛집 추천해줘.
0.3389 | 머리가 깨질 듯이 아프고 열이 나요.  <->  주식 시장이 오늘 폭락했습니다.
0.0644 | 머리가 깨질 듯이 아프고 열이 나요.  <->  강남역 근처 맛집 추천해줘.
0.0793 | 주식 시장이 오늘 폭락했습니다.  <->  강남역 근처 맛집 추천해줘.
